## 오라클 연동

- 보통 `Oracledb` 라이브러리 사용. 이전에는 `cx_Oracle`이라고 사용했음
- pip로 oracledb 설치
    ```bash
    pip install oracledb
    ```

In [3]:
# 설치
!pip install oracledb

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 49.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 34.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### oracle 연결 시작
- oracledb 라이브러리 소스에 추가
- 접속정보를 입력
- sql 쿼리 실행 확인

#### DB 연결 시 필요정보
- DB 접속정보 (Data Source Name)
- 연결 개체 (Connection)
- 커서 (Cursor)
- 트랜잭션 (Commit, Rollback)

#### SELECT 실행

In [24]:
import oracledb

In [25]:
oracledb.__version__

'3.4.2'

In [26]:
# DB접속정보 설정
# jdbc:oracle:thin:@//localhost:1521/XE 중에서 localhost 이후 부터 사용. ex. localhost:1521/XE
dsn = 'localhost:1521/XE'

conn = oracledb.connect(
    user='test',
    password='test12345',
    dsn=dsn
)

print('Oracle 연결 성공!', conn.version)

Oracle 연결 성공! 21.3.0.0.0


In [27]:
# 커서 생성
cursor = conn.cursor()

# sql문 실행
sql = """
SELECT dept_id, dept_name, loc
    FROM dept
ORDER BY dept_id
"""

# 실행
cursor.execute(sql)

# 결과 출력
for row in cursor:
    print(row)

(10, '개발부', 'BUSAN')
(20, '데이터부', 'SEOUL')
(30, '영업부', 'SEOUL')
(40, '인사부', 'BUSAN')


In [28]:
# 종료
cursor. close()
conn.close()

#### 일반적인 방법

- 접속 후
- 비지니스 트랜잭션 처리가 끝나면
- 접속 종료

In [45]:
# 전체 SELECT 실행 코드 단위
import oracledb

# DB접속정보 설정
# jdbc:oracle:thin:@//localhost:1521/XE 중에서 localhost 이후 부터 사용. ex. localhost:1521/XE
dsn = 'localhost:1521/XE'

conn = oracledb.connect(
    user='test',
    password='test12345',
    dsn=dsn
)

print('Oracle 연결 성공!', conn.version)

# 커서 생성
cursor = conn.cursor()

# sql문 실행
sql = """
SELECT dept_id, dept_name, loc
    FROM dept
ORDER BY dept_id
"""

# 실행
cursor.execute(sql)

# 결과 출력
for row in cursor:
    print(row)

# 종료
cursor. close()
conn.close()

print('Oracle 접속 종료.')

Oracle 연결 성공! 21.3.0.0.0
(10, '개발부', 'BUSAN')
(20, '데이터부', 'SEOUL')
(30, '영업부', 'SEOUL')
(40, '인사부', 'BUSAN')
Oracle 접속 종료.


#### INSERT 실행

- DBeaver에서 TEST 스키마에 SEQ_DEPT 시퀀스 생성

In [46]:
import oracledb

# DB접속정보 설정
# jdbc:oracle:thin:@//localhost:1521/XE 중에서 localhost 이후 부터 사용. ex. localhost:1521/XE
dsn = 'localhost:1521/XE'
conn = oracledb.connect(user='test', password='test12345', dsn=dsn)
print('Oracle 연결 성공!', conn.version)

try:
    # 커서 생성
    cursor = conn.cursor()

    # INSERT 쿼리 작성
    sql = """INSERT INTO dept (dept_id, dept_name, loc)
             VALUES(SEQ_DEPT.NEXTVAL, :1, :2) """

    # INSERT 쿼리 실행
    dept_name = '생산부' ; loc = 'KWANGJU'
    cursor.execute(sql, (dept_name, loc))
    # 트랜잭션 처리
    conn.commit()

    cursor.close()
except Exception as e:
    print('예외발생 ', e.args)
    conn.rollback()
finally:
    conn.close()

print('INSERT 완료')

Oracle 연결 성공! 21.3.0.0.0
INSERT 완료


#### WHERE 조건 실행

In [1]:
# 전체 SELECT 실행 코드 단위
import oracledb

# DB접속정보 설정
# jdbc:oracle:thin:@//localhost:1521/XE 중에서 localhost 이후 부터 사용. ex. localhost:1521/XE
dsn = 'localhost:1521/XE'

conn = oracledb.connect(user='test', password='test12345', dsn=dsn)
print('Oracle 연결 성공!', conn.version)

# 커서 생성
cursor = conn.cursor()

# sql문 실행
sql = """SELECT dept_id, dept_name, loc
           FROM dept
          where loc = :loc
          ORDER BY dept_id """

# 실행, 조건절 파라미터 추가
cursor.execute(sql, loc='SEOUL')
rows = cursor.fetchall() # 데이터양이 작을 때 한번에 모두 가져오기

# 결과 출력
for row in rows:
    print(row)

# 종료
cursor. close()
conn.close()

print('Oracle 접속 종료.')

Oracle 연결 성공! 21.3.0.0.0
(20, '데이터부', 'SEOUL')
(30, '영업부', 'SEOUL')
Oracle 접속 종료.


#### UPDATE 실행

In [48]:
import oracledb

# DB접속정보 설정
# jdbc:oracle:thin:@//localhost:1521/XE 중에서 localhost 이후 부터 사용. ex. localhost:1521/XE
dsn = 'localhost:1521/XE'
conn = oracledb.connect(user='test', password='test12345', dsn=dsn)
print('Oracle 연결 성공!', conn.version)

try:
    # 커서 생성
    cursor = conn.cursor()

    # INSERT 쿼리 작성
    sql = """UPDATE dept
                SET dept_name = :1
                  , loc = :2
            where dept_id = :3 """

    # INSERT 쿼리 실행
    dept_name = '자재부' 
    loc = 'INCHEON'
    dept_id = 50

    cursor.execute(sql, (dept_name, loc, dept_id))
    # 트랜잭션 처리
    conn.commit()

    cursor.close()
except Exception as e:
    print('예외발생 ', e.args)
    conn.rollback()
finally:
    conn.close()

print('UPDATE 완료')

Oracle 연결 성공! 21.3.0.0.0
UPDATE 완료


#### DELETE 실행

In [49]:
import oracledb

# DB접속정보 설정
# jdbc:oracle:thin:@//localhost:1521/XE 중에서 localhost 이후 부터 사용. ex. localhost:1521/XE
dsn = 'localhost:1521/XE'
conn = oracledb.connect(user='test', password='test12345', dsn=dsn)
print('Oracle 연결 성공!', conn.version)

try:
    # 커서 생성
    cursor = conn.cursor()

    # INSERT 쿼리 작성
    sql = """DELETE FROM dept
               where dept_id = :1 """
    deot_id = 50

    # INSERT 쿼리 실행
    cursor.execute(sql, (dept_id, ))
    print('삭제 행 : ', cursor.rowcount)

    # 트랜잭션 처리
    conn.commit()

    cursor.close()
except Exception as e:
    print('예외발생 ', e.args)
    conn.rollback()
finally:
    conn.close()

print('UPDATE 완료')

Oracle 연결 성공! 21.3.0.0.0
삭제 행 :  1
UPDATE 완료
